# fix_report

> Phase 13, Track A report: `fixLogMetrics` (asks, checks, checked-false from the turn logs), `fixSummaries` (`FIX_PLAN`: A1 pooled-share gap closed, A2 liar loss recovered and no-liar cost, A3 win-rate gap closed, mechanism columns), `writeFix` (report.md, provenance first) and `figFix`. All findings are exploratory.

In [ ]:
#| default_exp fix_report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
import re

import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from explRL.bench_grid import rowKey
from explRL.fix_stats import gapClosed, holmFamily, lossRecovered
from explRL.llm_figures import _save
from explRL.llm_report import NAN, _f, _sel, _stat

CHECK = re.compile(r'Your check: T\d+ x mod \d+ = -?\d+ \(from agent \d+\) is (true|false)\.')
KIND = dict(request='affordance', table='affordance', record='affordance', verify='affordance', factsFirst='affordance',
            evidence='instruction', warn='instruction', dissent='instruction')
COLORS = dict(affordance='tab:blue', instruction='tab:orange')
MECH = ('welfare', 'welfareHonest', 'pooledShare', 'sharePerHeld', 'invalid')
P11 = dict(clean=dict(study='clue', channel='blackboard'), liar=dict(study='clueLiar', channel='blackboard'))


def _action(x):
    try:
        a = json.loads(x.get('response') or '')
    except (TypeError, ValueError):
        return None
    return a.get('action') if isinstance(a, dict) else None


def fixLogMetrics(records):
    """Per row key, from clue-board turn logs: asks (valid asks: report ['ask', k]), checks (valid replies whose action
    is 'check'), checkedFalse (check lines in later observations that found a false clue), and the invalid tries of
    each new action (askInvalid, checkInvalid)."""
    out = {}
    for x in records:
        if x.get('kind', 'turn') != 'turn':
            continue
        d, a = out.setdefault(x['key'], dict(asks=0, checks=0, checkedFalse=0, askInvalid=0, checkInvalid=0)), _action(x)
        rep = x.get('report')
        d['asks'] += bool(x['valid'] and isinstance(rep, (list, tuple)) and len(rep) and rep[0] == 'ask')
        d['askInvalid'] += a == 'ask' and not x['valid']
        d['checks'] += a == 'check' and bool(x['valid'])
        d['checkInvalid'] += a == 'check' and not x['valid']
        d['checkedFalse'] += sum(m.group(1) == 'false' for m in CHECK.finditer(x.get('obs', '')))
    return out


def _bySeed(rows, metric):
    return {int(_f(r['seed'])): _f(r[metric]) for r in rows}


def _base(rows, p11, kind, N, metric, study):
    "Baseline values per seed: the new run's, else Phase 11's (clean: main grid; liar: liar grid)."
    new = _bySeed(_sel(rows, study=study, level='baseline' + '|liar' * (kind == 'liar'), N=N), metric)
    return {**_bySeed(_sel(p11, runner='llm', N=N, **P11[kind]), metric), **new}


def _mech(rs, logm, metrics):
    m = {k: _stat(_f(r.get(k, NAN)) for r in rs)['mean'] for k in metrics}
    if logm is not None:
        m.update({k: _stat(logm.get(rowKey(r), {}).get(k, NAN) for r in rs)['mean'] for k in ('asks', 'checks', 'checkedFalse')})
    return dict(m, runs=len(rs))


def fixSummaries(rows, p11rows=(), logMetrics=None, profileRows=(), study='fixA', profileStudy='profileA'):
    """FIX_PLAN: A1 pooled-share gap closed, A2 loss recovered and no-liar cost, A3 win-rate gap closed (bootstrap CIs
    over the pairing units, Holm within each primary family), and the mechanism columns."""
    rows, p11, logm = list(rows), list(p11rows), logMetrics or {}
    S = dict(A1={}, A2={}, A2cost={}, A3={}, mechanism=dict(A1={}, A2={}, A3={}))
    fx, pr = _sel(rows, study=study), _sel(list(profileRows), study=profileStudy)
    for N in sorted({int(_f(r['N'])) for r in fx if r['level'] in ('request', 'table', 'evidence')}):
        base = _base(rows, p11, 'clean', N, 'pooledShare', study)
        for a in ('request', 'table', 'evidence'):
            arm = _sel(fx, level=a, N=N)
            if arm:
                S['A1'][f'{a}|{N}'] = dict(gapClosed(_bySeed(arm, 'pooledShare'), base, 1.0), arm=a, N=N, kind=KIND[a])
    loss = lambda clean, liar: {s: clean[s] - liar[s] for s in sorted(set(clean) & set(liar))}
    bl = loss(_base(rows, p11, 'clean', 4, 'welfareHonest', study), _base(rows, p11, 'liar', 4, 'welfareHonest', study))
    for a in ('record', 'verify', 'warn'):
        clean, liar = _sel(fx, level=a, N=4), _sel(fx, level=a + '|liar', N=4)
        if clean and liar:
            S['A2'][a] = dict(lossRecovered(loss(_bySeed(clean, 'welfareHonest'), _bySeed(liar, 'welfareHonest')), bl),
                              arm=a, kind=KIND[a])
        if clean:
            w, b = _bySeed(clean, 'welfare'), _base(rows, p11, 'clean', 4, 'welfare', study)
            S['A2cost'][a] = _stat(w[s] - b[s] for s in sorted(set(w) & set(b)))
    for v in sorted({r['level'].split('|')[1] for r in pr}):
        base, solo = _bySeed(_sel(pr, level=f'baseline|{v}'), 'success'), _bySeed(_sel(pr, level=f'solo|{v}'), 'success')
        for a in ('factsFirst', 'table', 'dissent'):
            arm = _sel(pr, level=f'{a}|{v}')
            if arm:
                S['A3'][f'{a}|{v}'] = dict(gapClosed(_bySeed(arm, 'success'), base, solo), arm=a, variant=v, kind=KIND[a])
    holmFamily(S['A1'], [k for k in ('request|4', 'request|8', 'table|4', 'table|8') if k in S['A1']])
    holmFamily(S['A2'], [k for k in ('record', 'verify') if k in S['A2']])
    holmFamily(S['A3'], [f'{a}|{v}' for a in ('factsFirst', 'table') for v in ('classic', 'lone') if f'{a}|{v}' in S['A3']])
    for lv, N in sorted({(r['level'], int(_f(r['N']))) for r in fx}):
        a2 = lv.endswith('|liar') or lv in ('record', 'verify', 'warn')
        parts = ('A1', 'A2') if (lv, N) == ('baseline', 4) else ('A2',) if a2 else ('A1',)
        for part in parts:
            S['mechanism'][part][f'{lv}|{N}'] = _mech(_sel(fx, level=lv, N=N), logm, MECH)
    for lv in sorted({r['level'] for r in pr}):
        rs = _sel(pr, level=lv)
        S['mechanism']['A3'][lv] = dict(_mech(rs, None, ('success', 'uniquePosted', 'factsPosted', 'opinions', 'invalid')),
                                        votesAgainst=_stat(_f(r['N']) - _f(r['votesA']) for r in rs
                                                           if not lv.startswith('solo'))['mean'])
    return S


def _g(x, d=2):
    return 'n/a' if x is None or x != x else f'{x:.{d}f}'


def _row(k, v):
    return (f"| {k} | {v['kind']} | {_g(v['gap'])} [{_g(v['ci'][0])}, {_g(v['ci'][1])}] | {_g(v['base'])} | "
            f"{_g(v['ceiling'])} | {_g(v['diff']['mean'], 3)} | {v['n']} | {_g(v['p'], 3)} | "
            f"{_g(v['pHolm'], 3) if v['primary'] else 'control'} |")


TITLES = dict(A1=('A1: pooling on the clue board, pooled-share gap closed (ceiling 1; paired on board seed)', 'pooled share',
                  'gap closed'),
              A2=('A2: one liar, loss recovered (ceiling: no loss; paired on board seed)', 'honest loss', 'loss recovered'),
              A3=('A3: hidden-profile vote, win-rate gap closed (ceiling: solo; paired on episode seed)', 'win rate',
                  'gap closed'))


def writeFix(path, S, meta, parts=('A1', 'A2', 'A3')):
    "report.md: provenance first, then per part the gap closed per arm, the A2 no-liar costs and the mechanism (exploratory)."
    L = [f"# Phase 13, Track A: fixes for Claude teams ({meta['mode']})", '', 'Exploratory, pre-specified in `FIX_PLAN`.', '',
         f"- Plan hash `{meta['planHash']}`, git commit `{meta['commit']}`; model {meta['model']}; prices checked "
         f"{meta['pricesChecked']}; cost ${meta['cost']:.2f}; total API spend ${meta['spent']:.2f} of the ${meta['cap']:.0f} "
         f"cap; calls {meta['calls']}; wall time {meta['wall']:.0f} s"]
    for part in parts:
        title, base, stat = TITLES[part]
        L += ['', f'## {title}', '', f'| arm | kind | {stat} [95% CI] | baseline {base} | ceiling | mean paired difference '
              '| n | p | Holm p |', '|---|---|---|---|---|---|---|---|---|']
        L += [_row(k, v) for k, v in S[part].items()]
        if part == 'A2':
            L += ['', "No-liar cost (arm's welfare minus baseline's, no liar, paired on seed)", '',
                  '| arm | welfare difference [95% CI] | n |', '|---|---|---|']
            L += [f"| {k} | {_g(v['mean'], 3)} [{_g(v['ci'][0], 3)}, {_g(v['ci'][1], 3)}] | {v['n']} |"
                  for k, v in S['A2cost'].items()]
        mech = S['mechanism'][part]
        cols = sorted({c for v in mech.values() for c in v if c != 'runs'})
        if mech:
            L += ['', f'Mechanism ({part}, means per cell)', '', '| cell | runs | ' + ' | '.join(cols) + ' |',
                  '|---|---|' + '---|' * len(cols)]
            L += [f"| {k} | {v['runs']} | " + ' | '.join(_g(v.get(c, NAN)) for c in cols) + ' |' for k, v in mech.items()]
    path.write_text('\n'.join(L) + '\n')
    return path


def figFix(out, S, parts=('A1', 'A2', 'A3')):
    "fix.png: gap closed (A2: loss recovered) per arm with 95% intervals, one panel per part; affordances vs instructions."
    parts = [p for p in parts if S[p]]
    fig, axs = plt.subplots(1, max(len(parts), 1), figsize=(4.5 * max(len(parts), 1), 3.5), squeeze=False)
    for ax, part in zip(axs[0], parts):
        items = list(S[part].items())
        for y, (k, v) in enumerate(items):
            lo, hi = (np.nan_to_num(x, nan=v['gap']) for x in v['ci'])
            ax.errorbar([v['gap']], [y], xerr=[[max(v['gap'] - lo, 0)], [max(hi - v['gap'], 0)]], fmt='o', capsize=3,
                        color=COLORS[v['kind']])
        ax.set_yticks(range(len(items)), [k.replace('|', ' ') for k, _ in items], fontsize=8)
        for x, ls in ((0, '-'), (1, '--')):
            ax.axvline(x, color='k', lw=0.5, ls=ls)
        ax.invert_yaxis()
        ax.set(xlabel=TITLES[part][2], title=TITLES[part][0].split(',')[0])
    for kind, c in COLORS.items():
        axs[0][0].plot([], [], 'o', color=c, label=kind)
    axs[0][0].legend(fontsize=7, loc='lower right')
    return _save(fig, out / 'fix.png')

## Tests

In [ ]:
import math, tempfile
from pathlib import Path
from explRL.bench_grid import rowKey
from explRL.fix_report import *

# AC-5, synthetic rows with known effects. A1: request +0.3 pooled share, table +0, evidence +0.06 (same seed)
def row(study, level, N, seed, **m):
    return dict(study=study, factor='arm', level=level, channel='blackboard', N=N, seed=seed, budget='turns', episode=0,
                runner='llm', **m)
b = lambda s, N: 0.35 + 0.02 * (s % 4) + 0.05 * (N == 4)
wf = lambda s: 0.6 + 0.01 * (s % 3)
seeds = range(900, 910)
# Phase 11 rows: baseline for seeds 900-904 (main and liar grids), and a wrong value at 905 the new run must override
p11 = [row('clue', 'clue', N, s, pooledShare=b(s, N), welfare=wf(s), welfareHonest=wf(s)) for N in (4, 8) for s in range(900, 906)]
p11 = [dict(r, pooledShare=0.0) if r['seed'] == 905 else r for r in p11]
p11 += [row('clueLiar', 'liar', 4, s, welfareHonest=wf(s) - 0.4, nLiar=1) for s in range(900, 903)]
p11 += [dict(row('clue', 'clue', 4, s, pooledShare=0.9, welfare=0.0, welfareHonest=0.0), channel='messages') for s in seeds]
rows = [row('fixA', 'baseline', N, s, pooledShare=b(s, N), welfare=wf(s), welfareHonest=wf(s), sharePerHeld=0.2)
        for N in (4, 8) for s in range(905, 910)]
rows += [row('fixA', 'baseline|liar', 4, s, welfareHonest=wf(s) - 0.4, welfare=0.0) for s in range(903, 910)]
rows += [row('fixA', a, N, s, pooledShare=b(s, N) + d, welfare=wf(s), welfareHonest=wf(s), sharePerHeld=0.5)
         for a, d in (('request', 0.3), ('table', 0.0), ('evidence', 0.06)) for N in (4, 8) for s in seeds]
# A2: honest loss 0.1 (record), 0.4 (verify), 0.3 (warn) against the baseline's 0.4; record costs 0.05 with no liar
for a, loss, cost in (('record', 0.1, -0.05), ('verify', 0.4, 0.0), ('warn', 0.3, 0.0)):
    rows += [row('fixA', a, 4, s, welfare=wf(s) + cost, welfareHonest=wf(s) + cost, pooledShare=0.5) for s in seeds]
    rows += [row('fixA', a + '|liar', 4, s, welfare=0.0, welfareHonest=wf(s) + cost - loss) for s in seeds]
# A3: baseline wins 20% of episodes, the solo agent 90%, factsFirst 40% (same episodes), table 20%, dissent 30%
def prow(level, s, success, votesA=2):
    return row('profileA', level, 4, s, success=success, votesA=votesA, uniquePosted=0.5, factsPosted=6, opinions=2, invalid=0)
won = dict(baseline=lambda s: s % 5 == 0, factsFirst=lambda s: s % 5 in (0, 1), table=lambda s: s % 5 == 0,
           dissent=lambda s: s % 10 in (0, 5, 1), solo=lambda s: s % 10 != 9)
prof = [prow(f'{a}|{v}', s, int(f(s))) for a, f in won.items() for v in ('classic', 'lone') for s in range(5000, 5100)]
logm = {rowKey(r): dict(asks=3, checks=0, checkedFalse=0) for r in rows if r['level'] == 'request'}
S = fixSummaries(rows, p11, logm, prof)
mb = lambda N: sum(b(s, N) for s in seeds) / 10
for N in (4, 8):
    assert abs(S['A1'][f'request|{N}']['gap'] - 0.3 / (1 - mb(N))) < 1e-9 and S['A1'][f'request|{N}']['n'] == 10
    assert abs(S['A1'][f'table|{N}']['gap']) < 1e-9 and abs(S['A1'][f'evidence|{N}']['gap'] - 0.06 / (1 - mb(N))) < 1e-9
    v = S['A1'][f'request|{N}']
    assert v['ci'][0] <= v['gap'] <= v['ci'][1] and v['primary'] and v['kind'] == 'affordance'
assert not S['A1']['evidence|4']['primary'] and math.isnan(S['A1']['evidence|4']['pHolm']) and S['A1']['evidence|4']['kind'] == 'instruction'
assert S['A1']['request|4']['pHolm'] >= S['A1']['request|4']['p'] and S['A1']['table|4']['pHolm'] == 1.0
assert abs(S['A2']['record']['gap'] - 0.75) < 1e-9 and abs(S['A2']['verify']['gap']) < 1e-9 and abs(S['A2']['warn']['gap'] - 0.25) < 1e-9
assert S['A2']['record']['n'] == 10 and S['A2']['record']['primary'] and not S['A2']['warn']['primary']
assert abs(S['A2cost']['record']['mean'] + 0.05) < 1e-9 and abs(S['A2cost']['verify']['mean']) < 1e-9
for v in ('classic', 'lone'):
    assert abs(S['A3'][f'factsFirst|{v}']['gap'] - 0.2 / 0.7) < 1e-9 and S['A3'][f'table|{v}']['gap'] == 0
    assert abs(S['A3'][f'dissent|{v}']['gap'] - 0.1 / 0.7) < 1e-9 and S['A3'][f'factsFirst|{v}']['n'] == 100
    lo, hi = S['A3'][f'factsFirst|{v}']['ci']
    assert lo < 0.2 / 0.7 < hi and 0 < lo and S['A3'][f'factsFirst|{v}']['pHolm'] < 0.05
# mechanism: asks from the logs, shares per held clue, votes against the hidden best; the clean N 4 baseline in both parts
assert S['mechanism']['A1']['request|4']['asks'] == 3 and S['mechanism']['A1']['request|4']['sharePerHeld'] == 0.5
assert 'baseline|4' in S['mechanism']['A1'] and 'baseline|4' in S['mechanism']['A2'] and 'record|liar|4' in S['mechanism']['A2']
assert 'record|4' not in S['mechanism']['A1'] and math.isnan(S['mechanism']['A1']['baseline|4']['asks'])
assert S['mechanism']['A3']['baseline|classic']['votesAgainst'] == 2 and S['mechanism']['A3']['baseline|classic']['success'] == 0.2
# missing parts stay empty; rows read back from CSV (strings) give the same numbers
assert fixSummaries([], p11)['A1'] == {} and fixSummaries(rows, p11)['A3'] == {}
str_rows = [{k: str(v) for k, v in r.items()} for r in rows]
assert fixSummaries(str_rows, p11)['A2']['record']['gap'] == S['A2']['record']['gap']
with tempfile.TemporaryDirectory() as d:
    meta = dict(mode='quick', planHash='abc123', commit='x', model='claude-haiku-5-5', pricesChecked='2026-10-12', cost=0.0,
                spent=47.22, cap=150, calls=0, wall=1.0)
    text = writeFix(Path(d) / 'report.md', S, meta).read_text()
    assert text.index('abc123') < text.index('## A1') < text.index('## A2') < text.index('## A3') and 'Exploratory' in text
    assert '| arm | kind | loss recovered [95% CI] |' in text and '| request|4 | affordance | ' in text and '| warn | instruction |' in text and 'No-liar cost' in text and 'control |' in text
    assert '## A1' not in writeFix(Path(d) / 'r3.md', S, meta, ('A3',)).read_text()
    assert figFix(Path(d), S).stat().st_size > 0
    assert figFix(Path(d), dict(S, A1={}, A2={})).stat().st_size > 0

# fixLogMetrics on hand-made logs: valid asks, checks (valid replies with action 'check'), checks that found a false clue
import json
rec = lambda key, a, valid, report=None, obs='': dict(key=key, kind='turn', valid=valid, report=report, obs=obs,
                                                      response=json.dumps(dict(action=a, target='T1', answer=0, clue='', notes='')))
logs = [rec('k1', 'ask', True, ['ask', 1]), rec('k1', 'ask', False), rec('k1', 'check', True),
        rec('k1', 'wait', True, obs='Your check: T1 x mod 7 = 4 (from agent 2) is false.'),
        rec('k1', 'check', True, obs='Your check: T3 x mod 5 = 0 (from agent 0) is true.'), rec('k1', 'check', False),
        rec('k2', 'share', True, ['share', 1, 7, 4]), dict(key='k2', kind='beliefs', response='{}'),
        dict(rec('k2', 'wait', False), response='not json')]
assert fixLogMetrics(logs) == dict(k1=dict(asks=1, checks=2, checkedFalse=1, askInvalid=1, checkInvalid=1),
                                   k2=dict(asks=0, checks=0, checkedFalse=0, askInvalid=0, checkInvalid=0))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()